In [ ]:
import pandas as pd

In [ ]:
file_path = r"LD2011_2014.txt"

with open(file_path, "r", encoding="utf-8", errors="replace") as f:
    for i in range(10):
        print(i + 1, f.readline())

In [ ]:
with open(file_path, "r", encoding="utf-8", errors="replace") as f:
    for i in range(10):
        line = f.readline()
        print(i + 1, line.count(";"))

In [ ]:
df = pd.read_csv(
    file_path,
    sep=";",
    decimal=",",
    index_col=0,
    low_memory=False
)

In [ ]:
df

In [ ]:
df.index = pd.to_datetime(df.index)

summary = []

for col in df.columns:
    s = df[col].copy()
    
    mean_load = s.mean()
    max_load = s.max()
    min_load = s.min()
    std_load = s.std()
    
    load_factor = mean_load / max_load if max_load > 0 else np.nan
    cv = std_load / mean_load if mean_load > 0 else np.nan
    
    zero_share = (s == 0).mean()
    missing_share = s.isna().mean()
    
    # dzień / noc
    day_load = s[(s.index.hour >= 8) & (s.index.hour <= 18)].mean()
    night_load = s[(s.index.hour < 6)].mean()
    day_night_ratio = day_load / night_load if night_load > 0 else np.nan
    
    # dni robocze / weekend
    weekday_load = s[s.index.dayofweek < 5].mean()
    weekend_load = s[s.index.dayofweek >= 5].mean()
    weekday_weekend_ratio = weekday_load / weekend_load if weekend_load > 0 else np.nan
    
    summary.append({
        "profile": col,
        "mean_load": mean_load,
        "max_load": max_load,
        "load_factor": load_factor,
        "cv": cv,
        "zero_share": zero_share,
        "missing_share": missing_share,
        "day_night_ratio": day_night_ratio,
        "weekday_weekend_ratio": weekday_weekend_ratio
    })

profiles_summary = pd.DataFrame(summary)

profiles_summary.head()

In [ ]:
industrial_candidates = profiles_summary[
    (profiles_summary["missing_share"] < 0.01) &
    (profiles_summary["zero_share"] < 0.01) &
    (profiles_summary["load_factor"] > 0.35) &
    (profiles_summary["load_factor"] < 0.85) &
    (profiles_summary["cv"] < 1.0) &
    (profiles_summary["weekday_weekend_ratio"] > 1.05)
].copy()

industrial_candidates = industrial_candidates.sort_values(
    by=["load_factor", "mean_load"],
    ascending=[False, False]
)

industrial_candidates.head(20)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
top_profiles = industrial_candidates["profile"].head(5).tolist()

for profile in top_profiles:
    plt.figure(figsize=(14, 4))
    plt.plot(df.index, df[profile])
    plt.title(f"Profil odbiorcy: {profile}")
    plt.xlabel("Data")
    plt.ylabel("Pobór energii / mocy")
    plt.show()

In [ ]:
for profile in top_profiles:
    temp = df[[profile]].copy()
    temp["hour"] = temp.index.hour
    temp["dayofweek"] = temp.index.dayofweek
    
    avg_day = temp.groupby("hour")[profile].mean()
    
    plt.figure(figsize=(8, 4))
    plt.plot(avg_day.index, avg_day.values, marker="o")
    plt.title(f"Średni profil dobowy: {profile}")
    plt.xlabel("Godzina")
    plt.ylabel("Średni pobór")
    plt.grid(True)
    plt.show()

In [ ]:
for profile in top_profiles:
    temp = df[[profile]].copy()
    temp["dayofweek"] = temp.index.dayofweek
    
    avg_week = temp.groupby("dayofweek")[profile].mean()
    
    plt.figure(figsize=(8, 4))
    plt.plot(avg_week.index, avg_week.values, marker="o")
    plt.title(f"Średni profil tygodniowy: {profile}")
    plt.xlabel("Dzień tygodnia, 0=poniedziałek")
    plt.ylabel("Średni pobór")
    plt.grid(True)
    plt.show()

In [ ]:
selected_profile = "MT_253"

industrial_2014 = df[[selected_profile]].copy()
industrial_2014 = industrial_2014.rename(columns={selected_profile: "industrial_load_kW"})

# Upewniamy się, że indeks jest datą
industrial_2014.index = pd.to_datetime(industrial_2014.index)

# Wybieramy tylko rok 2014
industrial_2014 = industrial_2014[industrial_2014.index.year == 2014].copy()

industrial_2014.head()

In [ ]:
industrial_2025 = industrial_2014.copy()

industrial_2025.index = industrial_2025.index.map(
    lambda x: x.replace(year=2026)
)

industrial_2025 = industrial_2025.sort_index()

industrial_2025.head()

In [ ]:
industrial_2025["industrial_load_MW"] = industrial_2025["industrial_load_kW"] / 1000

target_peak_MW = 20

industrial_2025["industrial_load_MW_scaled"] = (
    industrial_2025["industrial_load_MW"]
    / industrial_2025["industrial_load_MW"].max()
    * target_peak_MW
)

industrial_2025["industrial_load_MWh_15min_scaled"] = (
    industrial_2025["industrial_load_MW_scaled"] * 0.25
)

industrial_2025.head()

In [ ]:
industrial_2025[[
    "industrial_load_kW",
    "industrial_load_MW_scaled",
    "industrial_load_MWh_15min_scaled"
]].describe()

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(industrial_2025.index, industrial_2025["industrial_load_MW_scaled"])
plt.title("Profil poboru energii elektrycznej odbiorcy przemysłowego")
plt.xlabel("Data")
plt.ylabel("Moc [MW]")
plt.grid(True)
plt.show()

In [ ]:
temp = industrial_2025.copy()
temp["hour"] = temp.index.hour

avg_day = temp.groupby("hour")["industrial_load_MW_scaled"].mean()

plt.figure(figsize=(9, 5))
plt.plot(avg_day.index, avg_day.values, marker="o")
plt.title("Średni profil dobowy odbiorcy przemysłowego")
plt.xlabel("Godzina")
plt.ylabel("Średnia moc [MW]")
plt.grid(True)
plt.show()

In [ ]:
temp = industrial_2025.copy()
temp["dayofweek"] = temp.index.dayofweek

avg_week = temp.groupby("dayofweek")["industrial_load_MW_scaled"].mean()

plt.figure(figsize=(9, 5))
plt.plot(avg_week.index, avg_week.values, marker="o")
plt.title("Średni profil tygodniowy odbiorcy przemysłowego")
plt.xlabel("Dzień tygodnia, 0=poniedziałek")
plt.ylabel("Średnia moc [MW]")
plt.grid(True)
plt.show()

In [ ]:
temp = industrial_2025.copy()
temp["month"] = temp.index.month

monthly_energy = temp.groupby("month")["industrial_load_MWh_15min_scaled"].sum()

plt.figure(figsize=(9, 5))
plt.bar(monthly_energy.index, monthly_energy.values)
plt.title("Miesięczne zużycie energii elektrycznej odbiorcy przemysłowego")
plt.xlabel("Miesiąc")
plt.ylabel("Energia [MWh]")
plt.grid(axis="y")
plt.show()

In [ ]:
load_duration = industrial_2025["industrial_load_MW_scaled"].sort_values(ascending=False).reset_index(drop=True)

plt.figure(figsize=(10, 5))
plt.plot(load_duration)
plt.title("Krzywa trwania obciążenia odbiorcy przemysłowego")
plt.xlabel("Krok 15-minutowy uporządkowany malejąco")
plt.ylabel("Moc [MW]")
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(industrial_2025["industrial_load_MW_scaled"], bins=50)
plt.title("Rozkład poboru mocy odbiorcy przemysłowego")
plt.xlabel("Moc [MW]")
plt.ylabel("Liczba obserwacji")
plt.grid(True)
plt.show()

In [ ]:
temp = industrial_2025.copy()
temp["hour"] = temp.index.hour
temp["dayofweek"] = temp.index.dayofweek

heatmap_data = temp.pivot_table(
    values="industrial_load_MW_scaled",
    index="hour",
    columns="dayofweek",
    aggfunc="mean"
)

plt.figure(figsize=(10, 6))
plt.imshow(heatmap_data, aspect="auto")
plt.colorbar(label="Średnia moc [MW]")
plt.title("Heatmapa poboru: godzina × dzień tygodnia")
plt.xlabel("Dzień tygodnia, 0=poniedziałek")
plt.ylabel("Godzina")
plt.show()

In [ ]:
industrial_2025.to_csv("industrial_load_schedule_2026_15min.csv")